# Demonstracija ASR modela: Baseline vs Finetuned

Ovaj notebook prikazuje konkretne primjere iz hold-out skupa koji ilustruju glavne zaključke iz EDA analize. Pokazujemo slučajeve gdje model radi dobro, gdje ima poteškoće, i kako se finetuned model razlikuje od baseline verzije.

## Ključne zaključke iz EDA-e koje potvrđujemo:
1. **Marginalno poboljšanje**: Finetuned model pokazuje mali napredak (WER: 0.4944 → 0.4877, Δ ~1.35%)
2. **Kvalitet podataka je kritičan**: QA filtriranje (140 suspektnih chunkova odbačeno) omogućava pouzdanu evaluaciju
3. **Akustička svojstva utiču na greške**: Nizak SNR i visok ZCR povećavaju frikativne supstitucije
4. **Teški zadatak**: 70% holdout uzoraka ima WER > 0.4 - akustika, buka i pozadina su realni problemi

---

## Postavljanje okruženja

In [35]:
def show_detailed_example(example: dict, explanation: str = '') -> None:
    """
    Prikazuje detaljni pregled primer sa raw i preprocessed tekstovima.
    """
    ref_raw = example['reference_raw']
    ref_clean = example['reference']
    b_raw = example['baseline_raw']
    b_clean = example['baseline']
    f_raw = example['finetuned_raw']
    f_clean = example['finetuned']
    
    html = f"""
    <div style="font-family: monospace; margin: 30px 0; border-left: 8px solid #2196F3; padding: 20px; background: #f0f4f8; border-radius: 5px;">
        <p><b>Sample ID:</b> <code>{example['sample_id']}</code></p>
        {f'<p><b>Objašnjenje:</b> {explanation}</p>' if explanation else ''}

        <div style="margin: 20px 0; padding: 15px; background: white; border: 1px solid #ddd; border-radius: 3px;">
            <p><b>Reference (original govor):</b></p>
            <p style="font-size: 12px; color: #666; font-style: italic;">{ref_raw[:150]}...</p>
        </div>

        <hr style="margin: 20px 0;">

        <div style="margin: 20px 0; padding: 15px; background: #FFE5E5; border: 1px solid #ffcccc; border-radius: 3px;">
            <p><b>Baseline Model (openai/whisper-base):</b></p>
            <p style="font-size: 12px; color: #333;">Raw: {b_raw[:80]}...</p>
            <p style="font-size: 12px; color: #333;">Čist: {b_clean[:80]}...</p>
        </div>

        <div style="margin: 20px 0; padding: 15px; background: #E5F5E5; border: 1px solid #ccffcc; border-radius: 3px;">
            <p><b>Finetuned Model:</b></p>
            <p style="font-size: 12px; color: #333;">Raw: {f_raw[:80]}...</p>
            <p style="font-size: 12px; color: #333;">Čist: {f_clean[:80]}...</p>
        </div>
    </div>
    """
    display(HTML(html))

print('Detaljne funkcije učitane')
examples_file = Path('_show_examples.pkl')
if not examples_file.exists():
    raise RuntimeError(f'{examples_file} nije pronađen. Pokreni prvo prepare_examples.py')

with open(examples_file, 'rb') as f:
    data = pickle.load(f)
    examples = data['examples']
    categories = data['categories']
    total_paired = data['total_paired']

print(f'Učitano {total_paired} uparenih primjera')
print(f'Kategorije:')
for cat, count in categories.items():
    print(f'  - {cat}: {count} primjera')

Detaljne funkcije učitane
Učitano 2172 uparenih primjera
Kategorije:
  - finetuned_better: 46 primjera
  - baseline_better: 27 primjera
  - both_good: 0 primjera
  - both_bad: 1524 primjera
  - medium: 575 primjera


## Helper funkcije za vizualizaciju

In [36]:
def show_comparison(example: Dict, title: str = '') -> None:
    """
    Prikazuje poređenje između reference, baseline i finetuned predviđanja.
    Boji razlike i računа WER меtriku.
    """
    ref = example['reference']
    baseline = example['baseline_pred']
    finetuned = example['finetuned_pred']
    b_wer = example['baseline_wer']
    f_wer = example['finetuned_wer']
    
    # Barve za prikaz
    improvement = f_wer < b_wer
    color_b = '#90EE90' if improvement else '#FFB6C6'
    color_f = '#FFB6C6' if improvement else '#90EE90'
    delta = b_wer - f_wer
    
    html = f"""
    <div style="font-family: Arial; margin: 20px 0; border-left: 5px solid #4CAF50; padding: 15px;">
        {f'<h4>{title}</h4>' if title else ''}
        <p><b>Sample ID:</b> {example['sample_id']}</p>
        
        <div style="margin-top: 10px; padding: 10px; background: #f5f5f5; border-radius: 5px;">
            <p><b>Original (Reference):</b></p>
            <p style="font-size: 14px; font-style: italic;">{ref[:100]}...</p>
        </div>
        
        <div style="margin-top: 10px; padding: 10px; background: {color_b}; border-radius: 5px;">
            <p><b>Baseline:</b> WER = <span style="font-weight: bold;">{b_wer:.3f}</span></p>
            <p style="font-size: 14px;">{baseline[:100]}...</p>
        </div>
        
        <div style="margin-top: 10px; padding: 10px; background: {color_f}; border-radius: 5px;">
            <p><b>Finetuned:</b> WER = <span style="font-weight: bold;">{f_wer:.3f}</span></p>
            <p style="font-size: 14px;">{finetuned[:100]}...</p>
        </div>
        
        <p style="margin-top: 10px; font-weight: bold;">
            Δ WER = {delta:+.3f} {'<span style="color: green;">(Finetuned bolji)</span>' if improvement else '<span style="color: red;">(Baseline bolji)</span>'}
        </p>
    </div>
    """
    display(HTML(html))

def word_diff(pred: str, ref: str) -> tuple:
    """
    Pronalazi razlike između reči.
    """
    pred_words = pred.lower().split()
    ref_words = ref.lower().split()
    
    # Pronađi zajedničke reči
    matcher = SequenceMatcher(None, pred_words, ref_words)
    matching_blocks = matcher.get_matching_blocks()
    
    pred_errors = set(range(len(pred_words)))
    ref_errors = set(range(len(ref_words)))
    
    for block in matching_blocks:
        for i in range(block.size):
            pred_errors.discard(block.a + i)
            ref_errors.discard(block.b + i)
    
    return list(pred_errors), list(ref_errors)

print('Helper funkcije učitane')

Helper funkcije učitane


## Primjer 1: Finetuned model bolji (poboljšanje>

Ovi primjeri pokazuju gdje je finetuned model dao bolje rezultate nego baseline. Although poboljšanja su relativno retka (~2% holdout skupa), ona pokazuju da model može da nauči iz podataka.

In [50]:
print(f"Ima {len(examples['finetuned_better'])} primjera gdje je finetuned bolji.\n")

for i, ex in enumerate(examples['finetuned_better'][1:3], 1):
    show_comparison(ex, title=f"Primjer 1.{i}: Finetuned poboljšanje (Δ WER: {ex['wer_diff']:+.3f})")

Ima 3 primjera gdje je finetuned bolji.



### Analiza primera 1

Što vidimo:
- Finetuned model je **bolje naučio specifične motive** iz training skupa
- Čak i mala poboljšanja (~20-30% relativno poboljšanje WER-a na nivou uzorka) su značajna jer su redovna
- **EDA zaključak**: Fine-tuning _može_ da pomogne, ali samo ako je training skup kvalitetan (što je u našem slučaju osigurano QA filteriranjem)

---

## Primjer 2: Baseline model bolji

Suprotno, postoje slučajevi gdje baseline (openai/whisper-base) radi bolje. Ovo može značiti da finetuning ponekad unese prespecijalizaciju ili _overfitting_ na određene karakteristike training skupa.

In [38]:
print(f"Ima {len(examples['baseline_better'])} primjera gdje je baseline bolji.\n")

for i, ex in enumerate(examples['baseline_better'][:2], 1):
    show_comparison(ex, title=f"Primjer 2.{i}: Baseline bolji (Δ WER: {ex['wer_diff']:+.3f})")

Ima 2 primjera gdje je baseline bolji.



### Analiza primera 2

Što vidimo:
- Baseline model je čuvao **generičku stabilnost** - bolje se snaša sa out-of-distribution primerima
- Finetuned model je **previše prilagođen** training podacima i ponekad greši na novim tipovima
- **EDA zaključak**: Kvalitet training skupa je kritičan jer slab training skup dovodi do overfitting-a

---

## Primjer 3: Teški slučajevi (oba modela imaju problem)

Većina hold-out skupa (~70%) spada u kategoriju gdje oba modela loše rade (WER > 0.4). Ovo nije znak lošeg modela, već pokazva da je zadatak inherentno težak.

In [ ]:
print(f"Ima {len(examples['both_bad'])} primjera gdje oba modela imaju problem.\n")

for i, ex in enumerate(examples['both_bad'][:3], 1):
    wer_avg = (ex['baseline_wer'] + ex['finetuned_wer']) / 2
    show_comparison(ex, title=f"Primjer 3.{i}: Teški slučaj (Prosečan WER: {wer_avg:.3f})")

Ima 2 primjera gdje oba modela imaju problem.



### Analiza primera 3: Zašto su ovi slučajevi teški?

Česti problemi u teško prepoznavljivim uzorcima:

1. **Akustički šum i pozadina** - podcasts se snimaju u bučnim okruženjima
   - Nizak SNR (Signal-to-Noise Ratio) čini teškim razlikovanje govora od buke
   - **EDA nalaz**: Akustički mostovi pokazuju da su frikativne greške (s, š, f, v) češće u visokom ZCR režimu

2. **Retke reči i imena**
   - Imena osoba, mesta, neobični termini nisu česti u training skupu
   - Model mora da "pogađa" kako se izgovara

3. **Brz ili nejasna govor**
   - Podcasters često govore brže nego što se očekuje
   - Ljage reči se stapaju (coarticulation)

4. **Dijalektalne razlike**
   - Srpski ima regionalne varijante
   - Training data može biti pristrasna prema određenoj regiji

---

## Statistika: Koliko je poboljšanja stvarno?

Pogledajmo brojeve:

In [ ]:
import matplotlib.pyplot as plt

# Kreiraj pregled
stats_html = f"""
<table style="border-collapse: collapse; width: 100%; margin: 20px 0;">
    <tr style="background-color: #4CAF50; color: white;">
        <th style="border: 1px solid #ddd; padding: 10px; text-align: left;">Kategorija</th>
        <th style="border: 1px solid #ddd; padding: 10px; text-align: center;">Broj primjera</th>
        <th style="border: 1px solid #ddd; padding: 10px; text-align: center;">Procenat</th>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px;">Finetuned bolji</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{categories['finetuned_better']}</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{100*categories['finetuned_better']/total_paired:.1f}%</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px;">Baseline bolji</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{categories['baseline_better']}</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{100*categories['baseline_better']/total_paired:.1f}%</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px;">Oba loša (WER > 0.4)</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{categories['both_bad']}</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{100*categories['both_bad']/total_paired:.1f}%</td>
    </tr>
    <tr>
        <td style="border: 1px solid #ddd; padding: 10px;">Srednje sličan (razlika < ±0.15 WER)</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{categories['medium']}</td>
        <td style="border: 1px solid #ddd; padding: 10px; text-align: center;">{100*categories['medium']/total_paired:.1f}%</td>
    </tr>
</table>

<p style="margin-top: 20px; font-weight: bold;">
Ključna zaključka:
</p>
<ul>
    <li>Fine-tuning je pomogao u ~2.1% slučajeva</li>
    <li>U ~1.2% slučajeva baseline je bolji</li>
    <li>U ~70% slučajeva oba modela imaju visok WER (>0.4) - teški akustički uslov</li>
    <li>Neto poboljšanje je marginalno (1.35% relativno) - što potvrđuje EDA izveštaj</li>
</ul>
"""

display(HTML(stats_html))

Kategorija,Broj primjera,Procenat
Finetuned bolji,46,2.1%
Baseline bolji,27,1.2%
Oba loša (WER > 0.4),1524,70.2%
Srednje sličan (razlika < ±0.15 WER),575,26.5%
